# 04 -- LGD model (loss given default) -- the key feature

**What this notebook does (plain English):** When a mortgage defaults, the lender
doesn't lose everything -- it sells the house and recovers most of the money.
**LGD** is the slice that is actually lost. Unlike a typical consumer-credit
project (where LGD is an assumption), here we model LGD from Freddie Mac's
**real, settled loss figures**. We use a simple **two-stage** model: the chance
of *any* loss, times the *size* of the loss when it happens.

**Headline result:** modelled LGD is roughly **double in the downturn** (~55%)
versus the calm year (~25%) -- a real, data-driven downturn LGD, which is the
single thing this project exists to demonstrate.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table; LGD is modelled ONLY on defaulted, disposed loans.
import pandas as pd
import numpy as np
from src.models import TwoStageLGD
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet')
disposed = base[base['disposed'] & base['lgd'].notna()].copy()
print('disposed defaults used for LGD:', len(disposed))

disposed defaults used for LGD: 6749


In [3]:
# Fit the two-stage LGD model (P(loss) x severity) and predict back on them.
lgd_model = TwoStageLGD().fit(disposed)
disposed['lgd_hat'] = lgd_model.predict(disposed)

In [4]:
# Compare observed vs modelled LGD, downturn (2007/2008) vs calm (2015).
disposed['regime'] = np.where(disposed['vintage_year'].isin([2007, 2008]), 'downturn (2007-08)', 'calm (2015)')
tbl = disposed.groupby('regime').agg(
    disposed_defaults=('lgd', 'size'),
    observed_lgd=('lgd', 'mean'),
    modelled_lgd=('lgd_hat', 'mean'),
).reset_index().round(4)

In [5]:
# Add an "all vintages" row and save as this notebook's result table.
overall = pd.DataFrame([{
    'regime': 'all', 'disposed_defaults': len(disposed),
    'observed_lgd': round(disposed['lgd'].mean(), 4),
    'modelled_lgd': round(disposed['lgd_hat'].mean(), 4),
}])
lgd_summary = pd.concat([tbl, overall], ignore_index=True)
save_csv(lgd_summary, 'output/04_lgd_model.csv')
lgd_summary

,regime,disposed_defaults,observed_lgd,modelled_lgd
0,calm (2015),136,0.2464,0.2325
1,downturn (2007-08),6613,0.5673,0.5664
2,all,6749,0.5608,0.5596


**Reading the table:** `observed_lgd` is what actually happened;
`modelled_lgd` is the two-stage model's fit. The downturn row sits roughly twice
as high as the calm row -- the **downturn LGD** a stress test needs. The model is
built only on loans that truly disposed, so every number is grounded in a real
settled loss.